In [1]:
# ============================================================
# Nelder-Mead blend for PS S6E3
# completed version using HC v13 model_dict paths
# ============================================================

In [2]:
import os
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score

In [3]:
# -----------------------------
# basic config
# -----------------------------
class CFG:
    TARGET = "Churn"
    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e3/train.csv"
    TEST_PATH  = "/kaggle/input/competitions/playground-series-s6e3/test.csv"

    NPY_ROOT = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
    OUTDIR   = Path("/kaggle/working/nm_blend")
    OUTDIR.mkdir(parents=True, exist_ok=True)

    RANDOM_STATE = 42
    N_RESTARTS = 40
    MAX_ITER = 4000

    # 最初は core10 だけ
    # 良ければ core11 → core12
    RUN_SET = "core12"

    USE_GPU_AUC = True

In [4]:
# -----------------------------
# device
# -----------------------------
DEVICE = "cuda" if (torch.cuda.is_available() and CFG.USE_GPU_AUC) else "cpu"
print("DEVICE =", DEVICE)

DEVICE = cpu


In [5]:
# -----------------------------
# load train/test
# -----------------------------
train = pd.read_csv(CFG.TRAIN_PATH)
test  = pd.read_csv(CFG.TEST_PATH)

if train[CFG.TARGET].dtype == object:
    y_np = train[CFG.TARGET].map({"Yes": 1, "No": 0}).values.astype(np.float32)
else:
    y_np = train[CFG.TARGET].values.astype(np.float32)

y_t = torch.tensor(y_np, dtype=torch.float32, device=DEVICE)

In [6]:
# -----------------------------
# fast auc
# -----------------------------
def fast_auc_torch(y_true: torch.Tensor, y_score: torch.Tensor) -> torch.Tensor:
    order = torch.argsort(y_score)
    y_sorted = y_true[order]
    auc = torch.sum(torch.cumsum(1 - y_sorted, dim=0) * y_sorted)
    n_pos = torch.sum(y_sorted)
    n_neg = len(y_sorted) - n_pos
    if n_pos <= 0 or n_neg <= 0:
        return torch.tensor(0.5, device=y_score.device)
    auc = auc / (n_pos * n_neg)
    return auc

In [7]:
# -----------------------------
# helpers
# -----------------------------
def load_pred_file(path: Path) -> np.ndarray:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        col = CFG.TARGET if CFG.TARGET in df.columns else df.columns[-1]
        return df[col].values.astype(np.float32).ravel()
    arr = np.load(path, allow_pickle=True)
    return np.asarray(arr, dtype=np.float32).ravel()

def check_len(name: str, oof: np.ndarray, pred: np.ndarray):
    if len(oof) != len(train):
        raise ValueError(f"{name}: len(oof)={len(oof)} != len(train)={len(train)}")
    if len(pred) != len(test):
        raise ValueError(f"{name}: len(pred)={len(pred)} != len(test)={len(test)}")

def score_np(y_true: np.ndarray, pred: np.ndarray) -> float:
    return float(roc_auc_score(y_true, pred))

In [8]:
# -----------------------------
# registry
# HC v13 model_dict から実ファイル名を反映
# -----------------------------
MODEL_REGISTRY = {
    # 主軸
    "xgb_raw_lrpred": {
        "label": "XGB blx RAW Full Bi/Tri-gramTE LRpred outer20 inner5 s42",
        "oof": CFG.NPY_ROOT / "oof_xgb_blx_full_ngramTE_o20i5_s42_lrpred_cpu_mean.npy",
        "pred": CFG.NPY_ROOT / "pred_xgb_blx_full_ngramTE_o20i5_s42_lrpred_cpu_mean.npy",
        "hc_w": 0.476403,
    },
    "realmlp_e4": {
        "label": "RealMLP Vldm as-is e4 n_ens32 ls_eps0.02 outer10 TEcv10 s42",
        "oof": CFG.NPY_ROOT / "oof_realmlp_vldm_as-is_e4_nens32_lseps0.02_o10_tecv10_s42.npy",
        "pred": CFG.NPY_ROOT / "pred_realmlp_vldm_as-is_e4_nens32_lseps0.02_o10_tecv10_s42.npy",
        "hc_w": 0.374317,
    },
    "cat_digit_stronger": {
        "label": "CAT blx DIGIT STRONGER Bi/Tri-gramTE OptunaA outer20 inner5 s42",
        "oof": CFG.NPY_ROOT / "oof_cat_blx_DIGIT_stronger_o20_s42_gpu_mean.npy",
        "pred": CFG.NPY_ROOT / "pred_cat_blx_DIGIT_stronger_o20_s42_gpu_mean.npy",
        "hc_w": 0.239947,
    },

    # strong side 補助
    "tabm_log": {
        "label": "TabM blx LOG Full Bi/Tri-gramTE tabm-mini-normal tk32e50 outer10 inner5 s42",
        "oof": CFG.NPY_ROOT / "oof_tabm_blxLOG_tabm-mini-normal_tk32_e50_o10i5_s42_gpu.npy",
        "pred": CFG.NPY_ROOT / "pred_tabm_blxLOG_tabm-mini-normal_tk32_e50_o10i5_s42_gpu.npy",
        "hc_w": 0.113257,
    },
    "xgb_log_optuna": {
        "label": "XGB blx LOG Optuna Full Bi/Tri-gramTE outer20 inner5 s42",
        "oof": CFG.NPY_ROOT / "oof_xgb_blx_log_o20i5_s42_gpu_optuna_mean.npy",
        "pred": CFG.NPY_ROOT / "pred_xgb_blx_log_o20i5_s42_gpu_optuna_mean.npy",
        "hc_w": 0.043113,
    },
    "xgb_ibmmix_te2stage": {
        "label": "XGB vld ibmMix Te2stage outer10 inner10 s77",
        "oof": CFG.NPY_ROOT / "oof_xgb_ibmMix_te2stage_outer10_inner10_s77_pl0.npy",
        "pred": CFG.NPY_ROOT / "pred_xgb_ibmMix_te2stage_outer10_inner10_s77_pl0.npy",
        "hc_w": 0.042134,
    },

    # 負補正
    "xgb_digit": {
        "label": "XGB blx DIGIT Full Bi/Tri-gramTE outer20 inner5 s42",
        "oof": CFG.NPY_ROOT / "oof_xgb_blx_digit_o20i5_s42_cpu_mean.npy",
        "pred": CFG.NPY_ROOT / "pred_xgb_blx_digit_o20i5_s42_cpu_mean.npy",
        "hc_w": -0.121347,
    },
    "lgb_full": {
        "label": "LGB blx Full Bi/Tri-gramTE Optuna5h outer20 inner5 s42",
        "oof": CFG.NPY_ROOT / "oof_lgb_blx_full_ngramTE_optuna5h_o20i5_s42_cpu.npy",
        "pred": CFG.NPY_ROOT / "pred_lgb_blx_full_ngramTE_optuna5h_o20i5_s42_cpu.npy",
        "hc_w": -0.088995,
    },
    "modernnca_light": {
        "label": "ModernNCA blx Light e2 outer2inner2 s42",
        "oof": CFG.NPY_ROOT / "oof_modernnca_blx_light_e2_o2i2_s42.npy",
        "pred": CFG.NPY_ROOT / "pred_modernnca_blx_light_e2_o2i2_s42.npy",
        "hc_w": -0.071352,
    },
    "tabm_log_nodigit": {
        "label": "TabM blx LOG noDIGIT Full Bi/Tri-gramTE tabm-mini-normal tk32e50 outer10 inner5 s42",
        "oof": CFG.NPY_ROOT / "oof_tabm_blxLOG_noDIGIT_tabm-mini-normal_tk32_e50_o10i5_s42_gpu.npy",
        "pred": CFG.NPY_ROOT / "pred_tabm_blxLOG_noDIGIT_tabm-mini-normal_tk32_e50_o10i5_s42_gpu.npy",
        "hc_w": -0.060841,
    },

    # 拡張候補
    "realmlp_strictcv": {
        "label": "RealMLP Vldm strictCV outer10 TEcv10 s42",
        "oof": CFG.NPY_ROOT / "oof_realmlp_vldm_strictcv_o10_tecv10_s42.npy",
        "pred": CFG.NPY_ROOT / "pred_realmlp_vldm_strictcv_o10_tecv10_s42.npy",
        "hc_w": -0.039600,
    },
    "tabm_raw": {
        "label": "TabM blx RAW Full Bi/Tri-gramTE tabm-mini-normal tk32e50 outer10 inner5 s42",
        "oof": CFG.NPY_ROOT / "oof_tabm_blx_tabm-mini-normal_tk32_e50_o10i5_s42_gpu.npy",
        "pred": CFG.NPY_ROOT / "pred_tabm_blx_tabm-mini-normal_tk32_e50_o10i5_s42_gpu.npy",
        "hc_w": 0.0,
    },
}

MODEL_SETS = {
    "core10": [
        "xgb_raw_lrpred",
        "realmlp_e4",
        "cat_digit_stronger",
        "tabm_log",
        "xgb_log_optuna",
        "xgb_ibmmix_te2stage",
        "xgb_digit",
        "lgb_full",
        "modernnca_light",
        "tabm_log_nodigit",
    ],
    "core11": [
        "xgb_raw_lrpred",
        "realmlp_e4",
        "cat_digit_stronger",
        "tabm_log",
        "xgb_log_optuna",
        "xgb_ibmmix_te2stage",
        "xgb_digit",
        "lgb_full",
        "modernnca_light",
        "tabm_log_nodigit",
        "realmlp_strictcv",
    ],
    "core12": [
        "xgb_raw_lrpred",
        "realmlp_e4",
        "cat_digit_stronger",
        "tabm_log",
        "xgb_log_optuna",
        "xgb_ibmmix_te2stage",
        "xgb_digit",
        "lgb_full",
        "modernnca_light",
        "tabm_log_nodigit",
        "realmlp_strictcv",
        "tabm_raw",
    ],
}

selected_names = MODEL_SETS[CFG.RUN_SET]
print("RUN_SET =", CFG.RUN_SET)
print("selected_names =", selected_names)

RUN_SET = core12
selected_names = ['xgb_raw_lrpred', 'realmlp_e4', 'cat_digit_stronger', 'tabm_log', 'xgb_log_optuna', 'xgb_ibmmix_te2stage', 'xgb_digit', 'lgb_full', 'modernnca_light', 'tabm_log_nodigit', 'realmlp_strictcv', 'tabm_raw']


In [9]:
# -----------------------------
# load selected models
# -----------------------------
oofs = {}
preds = {}
init_weights = []

for name in selected_names:
    info = MODEL_REGISTRY[name]
    oof = load_pred_file(info["oof"])
    pred = load_pred_file(info["pred"])
    check_len(name, oof, pred)

    auc = score_np(y_np, oof)
    print(f"{name:20s} | OOF AUC = {auc:.6f} | {info['label']}")

    oofs[name] = oof
    preds[name] = pred
    init_weights.append(float(info["hc_w"]))

oof_matrix_np = np.column_stack([oofs[n] for n in selected_names]).astype(np.float32)
pred_matrix_np = np.column_stack([preds[n] for n in selected_names]).astype(np.float32)

oof_matrix_t = torch.tensor(oof_matrix_np, dtype=torch.float32, device=DEVICE)
pred_matrix_t = torch.tensor(pred_matrix_np, dtype=torch.float32, device=DEVICE)

init_weights = np.array(init_weights, dtype=np.float32)

print("oof_matrix shape:", oof_matrix_np.shape)
print("pred_matrix shape:", pred_matrix_np.shape)

xgb_raw_lrpred       | OOF AUC = 0.919291 | XGB blx RAW Full Bi/Tri-gramTE LRpred outer20 inner5 s42
realmlp_e4           | OOF AUC = 0.919140 | RealMLP Vldm as-is e4 n_ens32 ls_eps0.02 outer10 TEcv10 s42
cat_digit_stronger   | OOF AUC = 0.918882 | CAT blx DIGIT STRONGER Bi/Tri-gramTE OptunaA outer20 inner5 s42
tabm_log             | OOF AUC = 0.918947 | TabM blx LOG Full Bi/Tri-gramTE tabm-mini-normal tk32e50 outer10 inner5 s42
xgb_log_optuna       | OOF AUC = 0.919253 | XGB blx LOG Optuna Full Bi/Tri-gramTE outer20 inner5 s42
xgb_ibmmix_te2stage  | OOF AUC = 0.918486 | XGB vld ibmMix Te2stage outer10 inner10 s77
xgb_digit            | OOF AUC = 0.918858 | XGB blx DIGIT Full Bi/Tri-gramTE outer20 inner5 s42
lgb_full             | OOF AUC = 0.919176 | LGB blx Full Bi/Tri-gramTE Optuna5h outer20 inner5 s42
modernnca_light      | OOF AUC = 0.913153 | ModernNCA blx Light e2 outer2inner2 s42
tabm_log_nodigit     | OOF AUC = 0.918442 | TabM blx LOG noDIGIT Full Bi/Tri-gramTE tabm-mini-norma

In [10]:
# -----------------------------
# HC baseline with the same subset
# -----------------------------
oof_hc = oof_matrix_np @ init_weights
hc_auc = score_np(y_np, oof_hc)
print(f"\nSubset HC baseline AUC: {hc_auc:.6f}")


Subset HC baseline AUC: 0.919779


In [11]:
# -----------------------------
# objective
# 弱いL2 penaltyだけ入れる
# -----------------------------
def neg_auc_free(weights_np: np.ndarray) -> float:
    w = np.asarray(weights_np, dtype=np.float32)
    penalty = 1e-5 * float(np.sum(w * w))

    if DEVICE == "cuda":
        w_t = torch.tensor(w, dtype=torch.float32, device=DEVICE)
        blend_t = oof_matrix_t @ w_t
        auc = fast_auc_torch(y_t, blend_t).item()
    else:
        blend = oof_matrix_np @ w
        auc = score_np(y_np, blend)

    return -(auc - penalty)

In [12]:
# -----------------------------
# run Nelder-Mead
# -----------------------------
np.random.seed(CFG.RANDOM_STATE)

best_w = None
best_auc = -1e9
trial_logs = []

for trial in range(CFG.N_RESTARTS):
    if trial < max(5, CFG.N_RESTARTS // 4):
        x0 = init_weights + np.random.randn(len(selected_names)).astype(np.float32) * 0.03
    elif trial < max(10, CFG.N_RESTARTS // 2):
        x0 = np.random.randn(len(selected_names)).astype(np.float32) * 0.10
    else:
        x0 = np.random.randn(len(selected_names)).astype(np.float32) * 0.20

    res = minimize(
        neg_auc_free,
        x0=x0,
        method="Nelder-Mead",
        options={
            "maxiter": CFG.MAX_ITER,
            "xatol": 1e-6,
            "fatol": 1e-6,
            "adaptive": True,
        },
    )

    cand_w = res.x.astype(np.float32)
    cand_oof = oof_matrix_np @ cand_w
    cand_auc = score_np(y_np, cand_oof)

    trial_logs.append({
        "trial": trial,
        "auc": cand_auc,
        "nit": getattr(res, "nit", None),
        "success": bool(res.success),
        "message": str(res.message),
    })

    if cand_auc > best_auc:
        best_auc = cand_auc
        best_w = cand_w.copy()

    if (trial == 0) or ((trial + 1) % 5 == 0):
        print(f"trial {trial+1:02d}/{CFG.N_RESTARTS} | this={cand_auc:.6f} | best={best_auc:.6f}")

if best_w is None:
    raise RuntimeError("Nelder-Mead failed to produce any solution.")

trial 01/40 | this=0.919810 | best=0.919810
trial 05/40 | this=0.919810 | best=0.919810
trial 10/40 | this=0.919810 | best=0.919810
trial 15/40 | this=0.919770 | best=0.919810
trial 20/40 | this=0.919810 | best=0.919810
trial 25/40 | this=0.919810 | best=0.919810
trial 30/40 | this=0.919807 | best=0.919810
trial 35/40 | this=0.919809 | best=0.919810
trial 40/40 | this=0.919810 | best=0.919810


In [13]:
# -----------------------------
# final score and outputs
# -----------------------------
oof_nm = oof_matrix_np @ best_w
pred_nm = pred_matrix_np @ best_w

nm_auc = score_np(y_np, oof_nm)
gain_vs_hc = nm_auc - hc_auc

print("\n" + "=" * 60)
print(f"Subset HC AUC    : {hc_auc:.6f}")
print(f"Nelder-Mead AUC  : {nm_auc:.6f}")
print(f"Gain vs HC       : {gain_vs_hc:+.6f}")
print("=" * 60)

print("\nOptimised weights:")
for name, w in sorted(zip(selected_names, best_w), key=lambda x: -abs(x[1])):
    if abs(w) >= 0.001:
        suffix = "  <-- negative" if w < 0 else ""
        print(f"{name:20s} {w:+.6f}{suffix}")

tag = f"nm_{CFG.RUN_SET}_r{CFG.N_RESTARTS}_it{CFG.MAX_ITER}_s{CFG.RANDOM_STATE}"

np.save(CFG.OUTDIR / f"oof_{tag}.npy", oof_nm.astype(np.float32))
np.save(CFG.OUTDIR / f"pred_{tag}.npy", pred_nm.astype(np.float32))

pd.DataFrame({
    "model_key": selected_names,
    "model_label": [MODEL_REGISTRY[n]["label"] for n in selected_names],
    "hc_init_weight": init_weights,
    "nm_weight": best_w,
    "abs_nm_weight": np.abs(best_w),
}).sort_values("abs_nm_weight", ascending=False).to_csv(
    CFG.OUTDIR / f"weights_{tag}.csv", index=False
)

pd.DataFrame(trial_logs).to_csv(CFG.OUTDIR / f"trial_log_{tag}.csv", index=False)

submission = pd.DataFrame({
    "id": test["id"].values,
    CFG.TARGET: pred_nm,
})
submission.to_csv(CFG.OUTDIR / f"submission_{tag}.csv", index=False)

summary = {
    "run_set": CFG.RUN_SET,
    "selected_names": selected_names,
    "selected_labels": [MODEL_REGISTRY[n]["label"] for n in selected_names],
    "hc_auc_subset": float(hc_auc),
    "nm_auc": float(nm_auc),
    "gain_vs_hc": float(gain_vs_hc),
    "n_restarts": CFG.N_RESTARTS,
    "max_iter": CFG.MAX_ITER,
    "random_state": CFG.RANDOM_STATE,
}
with open(CFG.OUTDIR / f"summary_{tag}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\nSaved:")
print(CFG.OUTDIR / f"oof_{tag}.npy")
print(CFG.OUTDIR / f"pred_{tag}.npy")
print(CFG.OUTDIR / f"weights_{tag}.csv")
print(CFG.OUTDIR / f"trial_log_{tag}.csv")
print(CFG.OUTDIR / f"submission_{tag}.csv")
print(CFG.OUTDIR / f"summary_{tag}.json")


Subset HC AUC    : 0.919779
Nelder-Mead AUC  : 0.919810
Gain vs HC       : +0.000031

Optimised weights:
realmlp_e4           +0.159644
cat_digit_stronger   +0.129142
xgb_log_optuna       +0.106760
xgb_raw_lrpred       +0.096351
xgb_digit            -0.059590  <-- negative
tabm_log             +0.052581
xgb_ibmmix_te2stage  +0.048972
lgb_full             -0.045638  <-- negative
tabm_raw             +0.041738
tabm_log_nodigit     -0.035259  <-- negative
modernnca_light      -0.026508  <-- negative
realmlp_strictcv     -0.025963  <-- negative

Saved:
/kaggle/working/nm_blend/oof_nm_core12_r40_it4000_s42.npy
/kaggle/working/nm_blend/pred_nm_core12_r40_it4000_s42.npy
/kaggle/working/nm_blend/weights_nm_core12_r40_it4000_s42.csv
/kaggle/working/nm_blend/trial_log_nm_core12_r40_it4000_s42.csv
/kaggle/working/nm_blend/submission_nm_core12_r40_it4000_s42.csv
/kaggle/working/nm_blend/summary_nm_core12_r40_it4000_s42.json
